# PC03 · DrawCAPTCHA: verificación humana mediante dibujo y CNN

**Curso:** Procesamiento de Imágenes · **Alumno:** _Nombre Apellido_ · **Código:** _XXXXXXXX_

## 1. Planteamiento del problema
Los CAPTCHA de texto distorsionado ya son resueltos por OCR y redes neuronales con más del 90 % de acierto, y los de selección de imágenes son molestos y poco accesibles en celular. Proponemos **DrawCAPTCHA**: el sistema pide al usuario **dibujar una figura elegida al azar** y una **red neuronal convolucional** verifica que el dibujo corresponde a lo pedido. Si coincide con suficiente confianza → **acceso concedido**; si no → **acceso denegado**.

**Categorías (5):** ⭐ estrella · ⚡ rayo · 🌀 espiral · ❤️ corazón · ☁️ nube

## 2. Justificación
* **Ciberseguridad / autenticación:** un bot debe *generar* un trazo que la red acepte para una figura aleatoria, lo que es más costoso que leer texto.
* **Usabilidad:** dibujar con el dedo es natural en celulares y tablets.
* **Métricas de seguridad reales:** además de la exactitud medimos **FAR** (tasa de falsa aceptación) y **FRR** (tasa de falso rechazo), igual que en sistemas biométricos.

## 3. Mejoras aplicadas a la red (se piden mínimo 3, aplicamos las 5)
| # | Mejora | Cómo se aplica |
|---|--------|----------------|
| 1 | Tamaño de entrada | 28×28 (base) → **64×64** |
| 2 | Más épocas | 10 (base) → hasta **80** con *EarlyStopping* |
| 3 | Modificar capas | BatchNorm, más filtros (32→256), GlobalAveragePooling, *data augmentation* |
| 4 | Dropout | 0.2 / 0.3 / 0.5 en distintas profundidades |
| 5 | Skip connections | **Bloques residuales tipo ResNet** |

## 4. Configuración e importaciones
> Recomendado: ejecutar en **Google Colab** con GPU (*Entorno de ejecución → Cambiar tipo → T4 GPU*).

In [ ]:
import os, glob, zipfile, random, base64, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

CLASES = ['estrella', 'rayo', 'espiral', 'corazon', 'nube']
SIZE = 64          # MEJORA 1: entrada 64x64 (la red base usa 28x28)
EPOCAS = 80        # MEJORA 2: más épocas (EarlyStopping detiene si deja de mejorar)
SEED = 42
SPLIT_POR_AUTOR = False   # True = probar con personas que la red NUNCA vio (más realista, requiere 4+ autores)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 5. Carga del dataset
El dataset se capturó con nuestra página web (`captura/index.html`). Comprime la carpeta `dataset` como **dataset.zip** y súbela cuando la celda lo pida.

In [ ]:
def buscar_dataset():
    for raiz, dirs, _ in os.walk('.'):
        if all(c in dirs for c in CLASES):
            return raiz
    return None

if buscar_dataset() is None:
    try:
        from google.colab import files
        print('Sube dataset.zip ...')
        subido = files.upload()
        for nombre in subido:
            if nombre.endswith('.zip'):
                zipfile.ZipFile(nombre).extractall('datos')
    except ImportError:
        print('No estás en Colab: coloca la carpeta dataset junto al notebook.')

RUTA = buscar_dataset()
print('Dataset encontrado en:', RUTA)
for c in CLASES:
    print(f'  {c:<9}', len(glob.glob(os.path.join(RUTA, c, '*.png'))), 'imágenes')

### 5.1 Preprocesamiento
1. Escala de grises e **inversión** (trazo blanco sobre fondo negro, como MNIST).
2. **Recorte al contenido** (*bounding box*) y centrado en un cuadrado con margen → la red no depende de dónde dibujó la persona.
3. Redimensionado a `SIZE×SIZE` y normalización a [0, 1].

In [ ]:
def preprocesar(img, size=SIZE):
    g = 255.0 - np.asarray(img.convert('L'), dtype=np.float32)
    ys, xs = np.where(g > 40)
    if len(xs) > 0:
        g = g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = g.shape
    lado = int(max(h, w) * 1.2) + 2                       # cuadrado con 10% de margen
    lienzo = np.zeros((lado, lado), dtype=np.float32)
    y0, x0 = (lado - h) // 2, (lado - w) // 2
    lienzo[y0:y0 + h, x0:x0 + w] = g
    im = Image.fromarray(lienzo.astype(np.uint8)).resize((size, size), Image.LANCZOS)
    return (np.asarray(im, dtype=np.float32) / 255.0)[..., None]

X, y, autores = [], [], []
for i, c in enumerate(CLASES):
    for f in sorted(glob.glob(os.path.join(RUTA, c, '*.png'))):
        X.append(preprocesar(Image.open(f)))
        y.append(i)
        partes = os.path.basename(f).split('_')
        autores.append(partes[1] if len(partes) >= 3 else 'desconocido')
X, y, autores = np.array(X), np.array(y), np.array(autores)
print('X:', X.shape, '| y:', y.shape, '| autores:', sorted(set(autores)))

### 5.2 Exploración visual del dataset

In [ ]:
fig, ejes = plt.subplots(len(CLASES), 8, figsize=(12, 7.5))
for i, c in enumerate(CLASES):
    idx = np.random.choice(np.where(y == i)[0], 8, replace=False)
    for j, k in enumerate(idx):
        ejes[i, j].imshow(X[k, ..., 0], cmap='gray'); ejes[i, j].axis('off')
    ejes[i, 0].set_title(c, loc='left', fontsize=11, fontweight='bold')
plt.suptitle(f'Muestras preprocesadas ({SIZE}x{SIZE})'); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
pd.Series(y).map(dict(enumerate(CLASES))).value_counts().reindex(CLASES).plot.bar(ax=ax[0], color='#2f6fed')
ax[0].set_title('Imágenes por clase'); ax[0].tick_params(axis='x', rotation=0)
pd.crosstab(autores, pd.Series(y).map(dict(enumerate(CLASES))).values).reindex(columns=CLASES).plot.bar(stacked=True, ax=ax[1])
ax[1].set_title('Imágenes por autor'); ax[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

### 5.3 División entrenamiento / validación / prueba (70 / 15 / 15)
Estratificada por clase. Con `SPLIT_POR_AUTOR = True` la prueba usa **personas distintas** a las del entrenamiento (escenario real de un CAPTCHA).

In [ ]:
idx = np.arange(len(y))
if SPLIT_POR_AUTOR and len(set(autores)) >= 4:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    idx_tr, idx_te = next(gss.split(idx, y, autores))
    idx_va, idx_te = train_test_split(idx_te, test_size=0.5, stratify=y[idx_te], random_state=SEED)
else:
    idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
    idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, stratify=y[idx_tmp], random_state=SEED)

X_tr, y_tr = X[idx_tr], y[idx_tr]
X_va, y_va = X[idx_va], y[idx_va]
X_te, y_te = X[idx_te], y[idx_te]
print(f'Entrenamiento: {len(y_tr)} | Validación: {len(y_va)} | Prueba: {len(y_te)}')

## 6. Modelo BASE (punto de partida, 28×28, 10 épocas)
Una CNN simple como la vista en clase. Sirve como **referencia** para demostrar cuánto aportan las mejoras.

In [ ]:
def a28(A):
    return tf.image.resize(A, (28, 28)).numpy()
X_tr28, X_va28, X_te28 = a28(X_tr), a28(X_va), a28(X_te)

base = keras.Sequential([
    layers.Input((28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(CLASES), activation='softmax'),
], name='CNN_base')
base.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
base.summary()
hist_base = base.fit(X_tr28, y_tr, validation_data=(X_va28, y_va), epochs=10, batch_size=32, verbose=2)

## 7. Modelo MEJORADO: ResNet pequeña con Dropout (64×64)
**Bloque residual (skip connection):** la entrada del bloque se suma a su salida, `y = F(x) + x`. Esto permite que el gradiente fluya directamente hacia capas anteriores, entrenar redes más profundas sin degradación y converger más rápido.

```
x ──► Conv3x3 ► BN ► ReLU ► Conv3x3 ► BN ──(+)──► ReLU ──►
 └───────────── atajo (Conv1x1 si cambia tamaño) ─┘
```

**Data augmentation** (solo en entrenamiento): rotación ±15°, zoom ±10 %, traslación ±8 % → simula la variabilidad de cómo dibuja cada persona.

In [ ]:
aumentacion = keras.Sequential([
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.08, 0.08),
], name='data_augmentation')

def bloque_residual(x, filtros, stride=1):
    atajo = x
    x = layers.Conv2D(filtros, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filtros, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or atajo.shape[-1] != filtros:              # ajusta tamaño del atajo
        atajo = layers.Conv2D(filtros, 1, strides=stride, use_bias=False)(atajo)
        atajo = layers.BatchNormalization()(atajo)
    x = layers.Add()([x, atajo])                                # SKIP CONNECTION
    return layers.ReLU()(x)

def crear_resnet(size=SIZE, n_clases=len(CLASES)):
    entrada = layers.Input((size, size, 1))
    x = aumentacion(entrada)
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = bloque_residual(x, 32)              # 64x64
    x = bloque_residual(x, 64, stride=2)    # 32x32
    x = layers.Dropout(0.2)(x)
    x = bloque_residual(x, 128, stride=2)   # 16x16
    x = layers.Dropout(0.3)(x)
    x = bloque_residual(x, 256, stride=2)   # 8x8
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    salida = layers.Dense(n_clases, activation='softmax')(x)
    return keras.Model(entrada, salida, name='DrawCAPTCHA_ResNet')

modelo = crear_resnet()
modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1),
    keras.callbacks.ModelCheckpoint('drawcaptcha_resnet.keras', monitor='val_accuracy', save_best_only=True),
]
hist = modelo.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                  epochs=EPOCAS, batch_size=32, callbacks=callbacks, verbose=2)

## 8. Curvas de entrenamiento

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for h, nombre, estilo in [(hist_base, 'Base 28x28', '--'), (hist, 'ResNet 64x64', '-')]:
    ax[0].plot(h.history['accuracy'], estilo, label=f'{nombre} · train')
    ax[0].plot(h.history['val_accuracy'], estilo, label=f'{nombre} · val')
    ax[1].plot(h.history['loss'], estilo, label=f'{nombre} · train')
    ax[1].plot(h.history['val_loss'], estilo, label=f'{nombre} · val')
ax[0].set_title('Exactitud (accuracy)'); ax[1].set_title('Pérdida (loss)')
for a in ax: a.set_xlabel('Época'); a.legend(fontsize=8); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 9. Evaluación en el conjunto de prueba y MATRIZ DE CONFUSIÓN

In [ ]:
pred_base = base.predict(X_te28, verbose=0).argmax(1)
prob_res = modelo.predict(X_te, verbose=0)
pred_res = prob_res.argmax(1)

def matriz(y_true, y_pred, titulo):
    cm = confusion_matrix(y_true, y_pred)
    cmn = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=ax[0], cmap='Blues', colorbar=False)
    ConfusionMatrixDisplay(cmn, display_labels=CLASES).plot(ax=ax[1], cmap='Blues', colorbar=False, values_format='.0%')
    ax[0].set_title(f'{titulo} · conteos'); ax[1].set_title(f'{titulo} · normalizada por fila')
    plt.tight_layout(); plt.show()

matriz(y_te, pred_base, 'CNN base 28x28')
matriz(y_te, pred_res, 'ResNet mejorada 64x64')
print(classification_report(y_te, pred_res, target_names=CLASES, digits=3))

### 9.1 Comparación base vs. mejorada

In [ ]:
tabla = pd.DataFrame([
    {'Modelo': 'CNN base', 'Entrada': '28x28', 'Épocas': len(hist_base.history['loss']),
     'Parámetros': base.count_params(), 'Dropout': 'No', 'Skip connections': 'No',
     'Accuracy prueba': accuracy_score(y_te, pred_base)},
    {'Modelo': 'ResNet mejorada', 'Entrada': f'{SIZE}x{SIZE}', 'Épocas': len(hist.history['loss']),
     'Parámetros': modelo.count_params(), 'Dropout': 'Sí', 'Skip connections': 'Sí',
     'Accuracy prueba': accuracy_score(y_te, pred_res)},
])
tabla.style.format({'Accuracy prueba': '{:.2%}', 'Parámetros': '{:,}'})

### 9.2 ¿En qué se equivoca la red?

In [ ]:
errores = np.where(pred_res != y_te)[0]
print(f'{len(errores)} errores de {len(y_te)} imágenes de prueba')
n = min(12, len(errores))
if n:
    fig, ejes = plt.subplots(1, n, figsize=(1.6 * n, 2.3), squeeze=False)
    for a, k in zip(ejes[0], errores[:n]):
        a.imshow(X_te[k, ..., 0], cmap='gray'); a.axis('off')
        a.set_title(f'real: {CLASES[y_te[k]]}\npred: {CLASES[pred_res[k]]}\n{prob_res[k].max():.0%}', fontsize=8)
    plt.tight_layout(); plt.show()

## 10. DrawCAPTCHA: flujo de acceso
Regla de decisión: se pide la clase *c*. **Acceso concedido** si la red predice *c* **y** su probabilidad ≥ `UMBRAL`.

* **FRR (falso rechazo):** un humano dibuja bien y es rechazado → molesta al usuario.
* **FAR (falsa aceptación):** se dibuja otra figura (o garabato de bot) y se acepta → riesgo de seguridad.

Simulamos intentos con el conjunto de prueba: *genuinos* (se pide la figura que realmente se dibujó) e *impostores* (se pide una figura distinta).

In [ ]:
def acceso(prob, clase_pedida, umbral):
    return prob.argmax() == clase_pedida and prob[clase_pedida] >= umbral

umbrales = np.linspace(0.2, 0.99, 80)
FAR, FRR = [], []
for u in umbrales:
    genuinos = [acceso(p, t, u) for p, t in zip(prob_res, y_te)]
    impostores = [acceso(p, c, u) for p, t in zip(prob_res, y_te) for c in range(len(CLASES)) if c != t]
    FRR.append(1 - np.mean(genuinos)); FAR.append(np.mean(impostores))
FAR, FRR = np.array(FAR), np.array(FRR)

UMBRAL = 0.70
k = np.abs(umbrales - UMBRAL).argmin()
plt.figure(figsize=(8, 4))
plt.plot(umbrales, FAR * 100, label='FAR · falsa aceptación')
plt.plot(umbrales, FRR * 100, label='FRR · falso rechazo')
plt.axvline(UMBRAL, color='gray', ls='--', label=f'umbral elegido = {UMBRAL}')
plt.xlabel('Umbral de confianza'); plt.ylabel('%'); plt.title('Seguridad vs. usabilidad del CAPTCHA')
plt.legend(); plt.grid(alpha=.3); plt.show()
print(f'Con umbral {UMBRAL}:  FAR = {FAR[k]:.2%}   FRR = {FRR[k]:.2%}')

### 10.1 Demo interactiva: ¡dibuja y verifica! (Colab)
Ejecuta la celda, dibuja la figura que se pide en el recuadro y pulsa **Verificar**. Vuelve a ejecutarla para otro intento.

In [ ]:
CANVAS_HTML = '''
<div style="font-family:Segoe UI,sans-serif;max-width:300px">
 <div style="background:#2f6fed;color:#fff;padding:10px 12px;border-radius:10px 10px 0 0">
  <small>Verificación humana</small><div style="font-size:22px;font-weight:800">Dibuja: __CLASE__</div></div>
 <canvas id="cv" width="280" height="280" style="border:2px solid #2f6fed;touch-action:none;background:#fff;display:block"></canvas>
 <button id="lim" style="padding:8px 14px;margin-top:6px">Limpiar</button>
 <button id="ok" style="padding:8px 14px;margin-top:6px;background:#2f6fed;color:#fff;border:0;border-radius:6px">Verificar</button>
</div>
<script>
var cv=document.getElementById('cv'), ctx=cv.getContext('2d'), down=false;
function fondo(){ctx.fillStyle='#fff';ctx.fillRect(0,0,280,280);}
fondo(); ctx.lineWidth=9; ctx.lineCap='round'; ctx.lineJoin='round'; ctx.strokeStyle='#000';
function pos(e){var r=cv.getBoundingClientRect();return [e.clientX-r.left,e.clientY-r.top];}
cv.onpointerdown=function(e){down=true;var p=pos(e);ctx.beginPath();ctx.moveTo(p[0],p[1]);ctx.lineTo(p[0]+0.1,p[1]);ctx.stroke();};
cv.onpointermove=function(e){if(!down)return;var p=pos(e);ctx.lineTo(p[0],p[1]);ctx.stroke();};
window.onpointerup=function(){down=false;};
document.getElementById('lim').onclick=fondo;
var dibujo=new Promise(function(res){document.getElementById('ok').onclick=function(){res(cv.toDataURL('image/png'));};});
</script>'''

def mostrar_resultado(img, prob, pedida):
    ok = acceso(prob, pedida, UMBRAL)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw={'width_ratios': [1, 2]})
    ax[0].imshow(img[..., 0], cmap='gray'); ax[0].axis('off'); ax[0].set_title('Entrada a la red')
    colores = ['#17a34a' if (i == pedida and ok) else '#dc3b3b' if i == prob.argmax() else '#9aa5bd' for i in range(len(CLASES))]
    ax[1].barh(CLASES, prob, color=colores); ax[1].axvline(UMBRAL, ls='--', color='gray'); ax[1].set_xlim(0, 1)
    ax[1].set_title(('ACCESO CONCEDIDO' if ok else 'ACCESO DENEGADO') + f'  (se pidió: {CLASES[pedida]})',
                    color='#17a34a' if ok else '#dc3b3b', fontweight='bold')
    plt.tight_layout(); plt.show()
    print('✅ Eres humano, puedes pasar.' if ok else '⛔ Verificación fallida, inténtalo de nuevo.')

def captcha():
    pedida = random.randrange(len(CLASES))
    try:
        from google.colab.output import eval_js
        from IPython.display import HTML, display
        display(HTML(CANVAS_HTML.replace('__CLASE__', CLASES[pedida].upper())))
        datos = eval_js('dibujo')
        img = preprocesar(Image.open(io.BytesIO(base64.b64decode(datos.split(',')[1]))))
    except ImportError:   # fuera de Colab: usa una imagen de prueba al azar
        k = random.randrange(len(X_te)); img = X_te[k]
        print('(Fuera de Colab) usando imagen de prueba de clase:', CLASES[y_te[k]])
    prob = modelo.predict(img[None], verbose=0)[0]
    mostrar_resultado(img, prob, pedida)

captcha()

## 11. Guardar el modelo

In [ ]:
modelo.save('drawcaptcha_resnet.keras')
print('Modelo guardado: drawcaptcha_resnet.keras')
# Para descargarlo desde Colab:
# from google.colab import files; files.download('drawcaptcha_resnet.keras')

## 12. Conclusiones
> ✍️ **Completa con tus resultados reales** (números de las celdas anteriores):

1. La CNN base (28×28, 10 épocas) obtuvo **__ %** de exactitud; la ResNet mejorada (64×64, Dropout, skip connections, augmentation) obtuvo **__ %**.
2. Según la matriz de confusión, las clases que más se confunden son **__ y __**, porque _(ej.: la espiral y la nube comparten trazos curvos)_.
3. Dropout + data augmentation redujeron el sobreajuste: la brecha entre train y val pasó de __ a __.
4. Con umbral 0.70 el CAPTCHA tiene FAR = __ % y FRR = __ %: _(interpretar el equilibrio seguridad/usabilidad)_.
5. **Trabajo futuro:** más dibujantes, clase "garabato/bot" para rechazar trazos aleatorios, evaluar ataques con imágenes generadas, uso en el navegador con TensorFlow.js.